# IMPORTS & GIT

In [ ]:
!pip install ultralytics einops catboost

In [ ]:
import os

# 1. Move to the root working directory first
%cd /kaggle/working

repo_path = '/kaggle/working/CV_project'
# 2. Delete the old folder safely
if os.path.exists(repo_path):
    print("Deleting existing folder...")
    !rm -rf {repo_path}

# 3. Get secrets and clone
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("github_token")

repo_url = f"https://{token}@github.com/CristianApost0L/CV_project.git"

!git clone -b stable-version {repo_url}

# 4. Now move into the new folder
%cd CV_project

!git log -1 --pretty=format:"Latest commit: %h - %s"

# MAIN LOOP

In [ ]:
!python main.py \
  --recognition-model 'HDGCN' \
  --path '/kaggle/input/tennis-rally-videos/input_video.mp4' \
  --window-size 40 \
  --player-detection-court-margin 50 \
  --output-path '/kaggle/working/output_videos'\
  --p1-handedness 'right' \
  --p2-handedness 'right'

# Video visualization

In [ ]:
import os
from IPython.display import Video, display

source_path = '/kaggle/working/output_videos/input_video.mp4'

display_path = '/kaggle/working/watchable_video.mp4'

# 3. Clean up previous display file if it exists
if os.path.exists(display_path):
    os.remove(display_path)

if os.path.exists(source_path):
    print(f"Converting {source_path} to H.264 for playback...")
    
    # -y overwrites without asking
    # -vcodec libx264 ensures browser compatibility
    !ffmpeg -y -i "{source_path}" -vcodec libx264 -crf 23 "{display_path}"

    print("Conversion complete. Displaying video:")
    display(Video(display_path, width=720, embed=True))
else:
    print(f"❌ File not found at: {source_path}")
    print("Check your output folder to see if the filename is different (e.g., 'processed_clip_1_0.mp4').")

# SIMPLE TESTER LOOP (logs only)

In [ ]:
import os, glob, json, subprocess

# --- CONFIGURATION ---
TARGET_FOLDER = "/kaggle/input/tennis-rally-videos" 
SCRIPT_PATH = "/kaggle/working/CV_project/main.py" 
OUTPUT_PATH = "/kaggle/working/testing-output"
# ---------------------

# create output dir if it doesn't exist
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 1. Find all Ground Truth JSON files
json_files = glob.glob(os.path.join(TARGET_FOLDER, "*.json"))

for file in json_files:
    print(file)

for json_path in json_files:
    # 2. Derive the video path (same name, but .mp4)
    video_path = json_path.replace(".json", ".mp4")
    
    if not os.path.exists(video_path):
        print(f"Skipping {json_path}: Corresponding video {video_path} not found.")
        continue

    json_name = os.path.basename(json_path)
    
    print(f"Processing: {os.path.basename(video_path)}...")
    
    # 3. Construct the command
    # Matches your manual command: !python main.py --path ...
    command = [
        "python", SCRIPT_PATH,
        "--recognition-model", 'CTRGCN',
        "--path", video_path,
        "--window-size", "40",
        "--player-detection-court-margin", "50",
        "--p1-handedness", 'right',
        "--p2-handedness", 'right'
    ]
    
    try:
        # 4. Run the script and capture Output
        # capture_output=True grabs both stdout (prints) and stderr (errors/tqdm)
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        
        # Combine stdout and stderr (tqdm often prints to stderr, but your logs are likely stdout)
        # We focus on stdout based on your example
        full_output = result.stdout
        
        # 5. Filter the output (Slice from the first relevant line)
        # We look for "[DEBUG]" or "Running Pose Estimation" to start saving
        start_markers = ["[DEBUG]", "Running Pose Estimation"]
        start_index = -1
        
        # Find the earliest occurrence of any marker
        for marker in start_markers:
            idx = full_output.find(marker)
            if idx != -1:
                if start_index == -1 or idx < start_index:
                    start_index = idx
        
        # If we found a start point, slice the string. Otherwise save everything.
        if start_index != -1:
            final_log = full_output[start_index:]
        else:
            final_log = full_output
            
        # 6. Save to .txt file
        video_filename = os.path.basename(video_path)
        txt_filename = video_filename.replace(".mp4", ".txt")
        output_txt_path = os.path.join(OUTPUT_PATH, txt_filename)
        
        with open(output_txt_path, "w", encoding="utf-8") as f:
            f.write(final_log)
            
        print(f"✅ Saved log to: {output_txt_path}")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Error running main.py for {video_path}")
        print("Error details:", e.stderr)

print("\nAll videos processed.")

# FULL TESTER LOOP 
shot type for each shot


player detection at two frames per video

In [ ]:
import os
import glob
import subprocess
import json
import cv2

# --- CONFIGURATION ---
INPUT_VIDEO_FOLDER = '/kaggle/input/tennis-rally-videos'
PLAYER_GT_FOLDER = '/kaggle/input/tennis-rally-videos/detection_ground_truth' 
SCRIPT_PATH = "main.py" 

# Outputs
OUTPUT_LOGS_DIR = "/kaggle/working/full_run_logs"
OUTPUT_VIDEOS_DIR = "/kaggle/working/output_videos"
DETECTIONS_DIR = "/kaggle/working/detections" 
SCREENSHOT_DIR = "/kaggle/working/validation_screenshots"

os.makedirs(OUTPUT_LOGS_DIR, exist_ok=True)
os.makedirs(OUTPUT_VIDEOS_DIR, exist_ok=True)
os.makedirs(DETECTIONS_DIR, exist_ok=True)
os.makedirs(SCREENSHOT_DIR, exist_ok=True)

# --- HELPER FUNCTIONS ---

print(f"🧹 Cleaning up old screenshots in: {SCREENSHOT_DIR}")
old_files = glob.glob(os.path.join(SCREENSHOT_DIR, "*.jpg"))
for f in old_files:
    os.remove(f)
print(f"   Removed {len(old_files)} old images.\n")

def is_point_in_box(point, bbox):
    """Checks if [x, y] is inside [x1, y1, x2, y2]"""
    px, py = point
    x1, y1, x2, y2 = bbox
    return x1 <= px <= x2 and y1 <= py <= y2

def validate_player_detection(frame_idx, gt_players, det_data, frame_img=None):
    """
    Checks if GT points exist in detected boxes for a specific frame index.
    """
    frame_boxes = det_data.get('frames', {}).get(str(frame_idx), [])
    p1_res = "N/A"
    p2_res = "N/A"
    
    # Validate P1
    if "p1" in gt_players:
        match = any(is_point_in_box(gt_players["p1"], b['bbox']) for b in frame_boxes)
        p1_res = "✅" if match else "❌"
        if frame_img is not None:
            color = (0, 255, 0) if match else (0, 0, 255) 
            cv2.circle(frame_img, (int(gt_players["p1"][0]), int(gt_players["p1"][1])), 10, color, -1)
            
    # Validate P2
    if "p2" in gt_players:
        match = any(is_point_in_box(gt_players["p2"], b['bbox']) for b in frame_boxes)
        p2_res = "✅" if match else "❌"
        if frame_img is not None:
            color = (0, 255, 0) if match else (0, 0, 255)
            cv2.circle(frame_img, (int(gt_players["p2"][0]), int(gt_players["p2"][1])), 10, color, -1)

    # Draw Boxes
    if frame_img is not None:
        for box in frame_boxes:
            x1, y1, x2, y2 = map(int, box['bbox'])
            cv2.rectangle(frame_img, (x1, y1), (x2, y2), (255, 0, 0), 2)

    return p1_res, p2_res

# --- MAIN EXECUTION ---

# 1. Select Files based on SHOT JSON existence (per your snippet)
json_files = glob.glob(os.path.join(INPUT_VIDEO_FOLDER, "*.json"))

video_queue = []
for json_path in json_files:
    # Derive video path
    video_path = json_path.replace(".json", ".mp4")
    if os.path.exists(video_path):
        video_queue.append(video_path)

video_queue.sort()

for file in video_queue:
    print(file)

print(f"Videos selected for processing: {len(video_queue)}\n")
print(f"{'VIDEO':<40} | {'START P1':<8} | {'START P2':<8} | {'END P1':<8} | {'END P2':<8}")
print("-" * 80)

for video_path in video_queue:
    filename = os.path.basename(video_path)
    file_root = os.path.splitext(filename)[0]
    
    # --- 1. RUN MAIN.PY (FULL VIDEO) ---
    command = [
        "python", SCRIPT_PATH,
        "--recognition-model", 'HDGCN',
        "--path", video_path,
        "--window-size", "40",
        "--player-detection-court-margin", "50",
        "--output-path", OUTPUT_VIDEOS_DIR
    ]
    
    try:
        # Capture output for saving
        result = subprocess.run(command, capture_output=True, text=True, check=True)
        full_output = result.stdout
        
        # --- 2. SAVE LOGS (Your requested logic) ---
        start_markers = ["[DEBUG]", "Running Pose Estimation", "Loading Global Ground Truth"]
        start_index = -1
        for marker in start_markers:
            idx = full_output.find(marker)
            if idx != -1:
                if start_index == -1 or idx < start_index: start_index = idx
        
        final_log = full_output[start_index:] if start_index != -1 else full_output
        
        txt_path = os.path.join(OUTPUT_LOGS_DIR, f"{file_root}.txt")
        with open(txt_path, "w", encoding="utf-8") as f: 
            f.write(final_log)

    except subprocess.CalledProcessError as e:
        print(f"{filename:<40} | ❌ CRASH (See error below)")
        # Save stderr if crash
        err_path = os.path.join(OUTPUT_LOGS_DIR, f"{file_root}_ERROR.txt")
        with open(err_path, "w") as f: f.write(e.stderr)
        continue

    # --- 3. VALIDATE PLAYER DETECTION ---
    player_gt_path = os.path.join(PLAYER_GT_FOLDER, f"{file_root}_ground_truth.json")
    
    s_p1, s_p2, e_p1, e_p2 = "N/A", "N/A", "N/A", "N/A"
    
    if os.path.exists(player_gt_path):
        with open(player_gt_path, 'r') as f: gt_data = json.load(f)
        
        # Load Detections generated by main.py
        det_path = os.path.join(DETECTIONS_DIR, f"{file_root}_detections.json")
        if os.path.exists(det_path):
            with open(det_path, 'r') as f: det_data = json.load(f)
            
            cap = cv2.VideoCapture(video_path)
            
            # Start Frame Check
            start_idx = gt_data.get("start_frame", 20)
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_idx)
            _, start_img = cap.read()
            s_p1, s_p2 = validate_player_detection(start_idx, gt_data.get("start_frame_players", {}), det_data, start_img)
            
            if start_img is not None:
                cv2.imwrite(os.path.join(SCREENSHOT_DIR, f"{file_root}_START.jpg"), start_img)

            # End Frame Check
            end_idx = gt_data.get("end_frame", 0)
            cap.set(cv2.CAP_PROP_POS_FRAMES, end_idx)
            _, end_img = cap.read()
            e_p1, e_p2 = validate_player_detection(end_idx, gt_data.get("end_frame_players", {}), det_data, end_img)
            
            if end_img is not None:
                cv2.imwrite(os.path.join(SCREENSHOT_DIR, f"{file_root}_END.jpg"), end_img)
            
            cap.release()
        else:
            s_p1 = "No Det"
    else:
        s_p1 = "No Player GT"

    # --- 4. PRINT SUMMARY ROW ---
    print(f"{filename:<40} | {s_p1:<8} | {s_p2:<8} | {e_p1:<8} | {e_p2:<8}")

print("-" * 100)
print(f"Full Logs saved to: {OUTPUT_LOGS_DIR}")
print(f"Screenshots saved to: {SCREENSHOT_DIR}")

# Watch validation screenshots

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

# Path to your screenshots
screenshot_dir = '/kaggle/working/validation_screenshots'

# 1. Identify unique video roots from filenames
# Files are named like: "my_video_START.jpg" or "my_video_END.jpg"
all_files = [f for f in os.listdir(screenshot_dir) if f.endswith(".jpg")]

roots = set()
for f in all_files:
    if f.endswith("_START.jpg"):
        roots.add(f[:-10]) # Remove "_START.jpg"
    elif f.endswith("_END.jpg"):
        roots.add(f[:-8])  # Remove "_END.jpg"

sorted_roots = sorted(list(roots))

if not sorted_roots:
    print("⚠️ No screenshots found! Check if the previous script actually saved them.")
else:
    print(f"Found {len(sorted_roots)} video pairs. Displaying...")

    # Grid Settings: 2 Columns (Start vs End), Rows = Number of Videos
    cols = 2 
    rows = len(sorted_roots)
    
    # Dynamic Figure Size
    plt.figure(figsize=(24, 6 * rows))

    for i, root in enumerate(sorted_roots):
        # Construct specific paths
        start_path = os.path.join(screenshot_dir, f"{root}_START.jpg")
        end_path = os.path.join(screenshot_dir, f"{root}_END.jpg")
        
        # --- LEFT COLUMN: START FRAME ---
        plt.subplot(rows, cols, 2*i + 1)
        if os.path.exists(start_path):
            img = cv2.imread(start_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.imshow(img)
            plt.title(f"{root}\nSTART (Frame 20)", fontsize=16, fontweight='bold')
        else:
            plt.text(0.5, 0.5, "Missing START Image", ha='center', fontsize=14)
        plt.axis('off')

        # --- RIGHT COLUMN: END FRAME ---
        plt.subplot(rows, cols, 2*i + 2)
        if os.path.exists(end_path):
            img = cv2.imread(end_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.imshow(img)
            plt.title(f"{root}\nEND (Frame -20)", fontsize=16, fontweight='bold')
        else:
            plt.text(0.5, 0.5, "Missing END Image", ha='center', fontsize=14)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# ANALYZE RESULTS

In [ ]:
import os
import glob
import re
from collections import defaultdict

# ================= CONFIGURATION =================
LOGS_FOLDER = '/kaggle/working/testing-output' 
# =================================================

def parse_and_aggregate_stats(folder_path):
    # --- Regex Patterns ---
    patterns = {
        # Matches summary lines
        'player_acc': re.compile(r"Player ID Accuracy:\s+(\d+)/(\d+)"),
        'shot_exact': re.compile(r"Shot Exact Matches:\s+(\d+)/(\d+)"),
        'shot_partial': re.compile(r"Shot Partial Matches:\s+(\d+)/(\d+)"),
        
        # Matches: Frame X ... [STATUS] ... | Expected: [shot_type] (P[12])
        # Group 1 = Status (PERFECT, PARTIAL, WRONG SHOT, WRONG PLAYER)
        # Group 2 = Shot Type (e.g., forehand_openstands)
        # Group 3 = Player ID (P1 or P2)
        'shot_detail': re.compile(r"Frame\s+\d+\s+:\s+.*?(PERFECT|PARTIAL|WRONG SHOT|WRONG PLAYER).*?\|\s+Expected:\s+([a-zA-Z0-9_]+)\s+\((P\d)\)")
    }

    # --- Storage ---
    global_stats = {
        'files_processed': 0,
        'player_correct': 0, 'player_total': 0,
        'exact_correct': 0,  'exact_total': 0,
        'partial_correct': 0, 'partial_total': 0
    }

    length_breakdown = defaultdict(lambda: {'files': 0, 'p_corr': 0, 'p_tot': 0, 'e_corr': 0, 'e_tot': 0, 'useful_corr': 0})
    
    # NEW STRUCTURE: { 'P1': { 'forehand': {...}, 'backhand': {...} }, 'P2': ... }
    player_shot_stats = {
        'P1': defaultdict(lambda: {'total': 0, 'exact': 0, 'partial': 0}),
        'P2': defaultdict(lambda: {'total': 0, 'exact': 0, 'partial': 0})
    }

    file_rankings = []

    # --- Processing ---
    files = glob.glob(os.path.join(folder_path, "*.txt"))
    
    if not files:
        print(f"❌ No .txt files found in '{folder_path}'")
        return

    print(f"📂 Found {len(files)} log files. Parsing...\n")

    for file_path in files:
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                global_stats['files_processed'] += 1
                filename = os.path.basename(file_path)

                # 1. Extract Summary Stats
                p_corr, p_tot = 0, 0
                e_corr, e_tot = 0, 0
                part_corr, part_tot = 0, 0

                p_match = patterns['player_acc'].search(content)
                if p_match:
                    p_corr, p_tot = int(p_match.group(1)), int(p_match.group(2))
                    global_stats['player_correct'] += p_corr
                    global_stats['player_total'] += p_tot

                e_match = patterns['shot_exact'].search(content)
                if e_match:
                    e_corr, e_tot = int(e_match.group(1)), int(e_match.group(2))
                    global_stats['exact_correct'] += e_corr
                    global_stats['exact_total'] += e_tot

                part_match = patterns['shot_partial'].search(content)
                if part_match:
                    part_corr, part_tot = int(part_match.group(1)), int(part_match.group(2))
                    global_stats['partial_correct'] += part_corr
                    global_stats['partial_total'] += part_tot
                
                # 2. Extract Per-Shot Breakdown Separated by Player
                shot_matches = patterns['shot_detail'].findall(content)
                
                for status, shot_type, player_id in shot_matches:
                    # Access the specific player's shot dictionary
                    if player_id in player_shot_stats:
                        stats = player_shot_stats[player_id][shot_type]
                        
                        stats['total'] += 1
                        
                        if "PERFECT" in status:
                            stats['exact'] += 1
                        elif "PARTIAL" in status:
                            stats['partial'] += 1
                        # WRONG SHOT/PLAYER contribute to total but not exact/partial

                # 3. Per File Calculations
                rally_len = e_tot 
                useful_hits = e_corr + part_corr
                
                if rally_len > 0:
                    length_breakdown[rally_len]['files'] += 1
                    length_breakdown[rally_len]['p_corr'] += p_corr
                    length_breakdown[rally_len]['p_tot'] += p_tot
                    length_breakdown[rally_len]['e_corr'] += e_corr
                    length_breakdown[rally_len]['e_tot'] += e_tot
                    length_breakdown[rally_len]['useful_corr'] += useful_hits

                p_pct = (p_corr / p_tot * 100) if p_tot > 0 else 0
                e_pct = (e_corr / e_tot * 100) if e_tot > 0 else 0
                u_pct = (useful_hits / e_tot * 100) if e_tot > 0 else 0

                file_rankings.append({
                    'name': filename,
                    'length': rally_len,
                    'p_acc': p_pct,
                    'e_acc': e_pct,
                    'u_acc': u_pct
                })

        except Exception as e:
            print(f"⚠️ Error reading {os.path.basename(file_path)}: {e}")

    # --- Calculation Helper ---
    calc_pct = lambda num, den: (num / den * 100) if den > 0 else 0.0

    # Global Calculations
    g_p_acc = calc_pct(global_stats['player_correct'], global_stats['player_total'])
    g_e_acc = calc_pct(global_stats['exact_correct'], global_stats['exact_total'])
    total_useful_hits = global_stats['exact_correct'] + global_stats['partial_correct']
    g_useful_acc = calc_pct(total_useful_hits, global_stats['exact_total'])

    # --- Output Phase ---
    print("="*80)
    print(f"📊 GLOBAL AGGREGATE REPORT ({global_stats['files_processed']} files)")
    print("="*80)
    print(f"🎯 Player ID Accuracy : {global_stats['player_correct']}/{global_stats['player_total']} ({g_p_acc:.1f}%)")
    print(f"🎯 Shot Exact Matches : {global_stats['exact_correct']}/{global_stats['exact_total']} ({g_e_acc:.1f}%)")
    print(f"🚀 TOTAL USEFUL SHOTS : {total_useful_hits}/{global_stats['exact_total']} ({g_useful_acc:.1f}%)")
    
    # ---------------- NEW SECTION: SHOT TYPE BY PLAYER ----------------
    print("\n" + "="*80)
    print("🎾 DETAILED SHOT BREAKDOWN BY PLAYER")
    print("="*80)
    
    for player_id in ['P1', 'P2']:
        print(f"\n👤 PLAYER {player_id[-1]} STATISTICS")
        print(f"{'Shot Type':<25} | {'Total':<6} | {'Exact':<8} | {'Partial':<8} | {'Useful %':<8}")
        print("-" * 70)
        
        # Get stats for this player
        p_shots = player_shot_stats[player_id]
        
        # Sort by total occurrences (most frequent shots first)
        sorted_shots = sorted(p_shots.items(), key=lambda x: x[1]['total'], reverse=True)
        
        if not sorted_shots:
            print(f"{'No shots recorded for this player':<60}")
            continue

        for shot, stats in sorted_shots:
            s_total = stats['total']
            s_exact = stats['exact']
            s_part = stats['partial']
            s_useful = s_exact + s_part
            
            useful_pct = calc_pct(s_useful, s_total)
            
            exact_str = f"{s_exact} ({calc_pct(s_exact, s_total):.0f}%)"
            part_str = f"{s_part} ({calc_pct(s_part, s_total):.0f}%)"
            
            print(f"{shot:<25} | {s_total:<6} | {exact_str:<8} | {part_str:<8} | {useful_pct:.1f}%")

    # ---------------- EXISTING SECTIONS ----------------
    print("\n" + "="*80)
    print("📈 BREAKDOWN BY RALLY LENGTH")
    print("="*80)
    print(f"{'Rally Len':<12} | {'Files':<5} | {'Player Acc':<12} | {'Exact Shot':<12} | {'Useful Shot':<12}")
    print("-" * 70)

    sorted_lengths = sorted(length_breakdown.keys())
    for length in sorted_lengths:
        data = length_breakdown[length]
        l_p_acc = calc_pct(data['p_corr'], data['p_tot'])
        l_e_acc = calc_pct(data['e_corr'], data['e_tot'])
        l_u_acc = calc_pct(data['useful_corr'], data['e_tot'])
        print(f"{length:<12} | {data['files']:<5} | {l_p_acc:>9.1f}%  | {l_e_acc:>9.1f}%  | {l_u_acc:>9.1f}%")

    print("\n" + "="*80)
    print("🏆 FILE RANKINGS (Top 5 Best & Worst)")
    print("="*80)

    def print_ranking(title, key):
        sorted_files = sorted(file_rankings, key=lambda x: x[key], reverse=True)
        print(f"\n🔹 {title}")
        print(f"{'Filename':<40} | {'Length':<8} | {'Accuracy':<10}")
        print("-" * 55)
        for entry in sorted_files:
            print(f"{entry['name']:<40} | {entry['length']:<8} | {entry[key]:.1f}%")

    print_ranking("Best Player Accuracy", 'p_acc')
    print_ranking("Best Shot Exact Matches", 'e_acc')
    print_ranking("Best Total Useful Shots", 'u_acc')

# Run the function
parse_and_aggregate_stats(LOGS_FOLDER)